In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Italy Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'

In [4]:
YEAR = 2023
MONTH = 'August'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,population,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,11.79324,45.35865,2023-08-16,veneto,abano terme,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,"19,349",46.86670,0.384349,0.071285,-0.348903,-0.071285,0.381060,0.079529,-0.350111,-0.079529,0.133306,0.081959,0.098001,0.081959,23.310,29.81,16.810,7.626000,0.600000,8.080000,-0.791538,18.365714,6.326364,21.500769,6.950000,0.0,99.583106,1412.123245,8982.126981,2074.539971,3,170.409233,10.722186,179.966181,0.0,1.063084,31,98.0,36,98.0,30,98.0,12,12,1,6,7,2,0,35,0,0
1,12.04199,45.06525,2023-08-16,veneto,adria,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,"20,233",46.64640,0.445564,0.125397,-0.401029,-0.125397,0.514870,0.157596,-0.469637,-0.157596,0.122878,0.106699,0.089933,0.106699,22.308,27.91,16.706,6.377211,0.808571,8.476395,-1.590923,16.728553,4.271543,21.836459,5.637644,0.0,52.918090,1541.629844,6701.380530,1644.125334,2,145.603087,-1.354496,180.070371,0.0,1.206604,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,21,0,0
2,10.77673,45.55680,2023-08-16,veneto,affi,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,"2,297",46.81410,0.359280,0.064787,-0.346506,-0.064787,0.475644,0.123431,-0.422465,-0.123431,0.123713,0.055039,0.094677,0.055039,18.580,19.17,17.990,5.821667,0.738000,8.631538,0.720667,17.207647,3.983750,18.918421,6.878889,0.0,126.245919,1370.553249,5306.025387,738.794180,6,207.641691,201.821006,178.733735,0.0,1.815834,31,84.0,30,84.0,30,84.0,10,10,1,6,6,2,0,43,0,0
3,11.96539,45.17531,2023-08-16,veneto,agna,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,"3,400",46.73306,0.332803,0.005401,-0.341340,-0.005401,0.517922,0.160751,-0.470167,-0.160751,0.177918,0.145539,0.130420,0.145539,21.440,27.35,15.530,4.870000,0.610000,7.752222,-2.034615,15.791176,4.124000,21.578571,5.119231,0.0,53.177647,1489.792309,16631.850238,1165.197945,3,121.193871,0.122498,180.188821,0.0,3.769479,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,29,0,0
4,12.04755,46.30297,2023-08-16,veneto,agordo,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,"4,249",47.84463,0.626855,0.269194,-0.556457,-0.269194,0.740179,0.310919,-0.623949,-0.310919,0.045194,0.019253,0.030461,0.019253,NaN,NaN,NaN,1.921429,-4.077692,5.623333,-2.298000,9.285385,-0.998333,11.544545,0.444545,0.0,0.000000,1341.774334,15023.566702,684.464446,18,124.001040,1470.386475,152.701532,0.0,2.289867,15,91.0,10,97.0,10,97.0,5,5,7,1,1,2,0,14,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)


In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,11.73897,44.96915,polesella,16,8,2023,0.991146
1,12.62107,45.75752,motta da livenza,16,8,2023,0.969150
2,12.64550,45.53937,jesolo,16,8,2023,0.964405
3,11.87982,45.22842,conselve,16,8,2023,0.963664
4,12.22945,45.56416,mogliano veneto,16,8,2023,0.962215
...,...,...,...,...,...,...,...
566,11.39118,45.89987,rotzo,16,8,2023,0.000003
567,11.93045,46.43842,rocca pietore,16,8,2023,0.000003
568,11.85565,46.37360,falcade,16,8,2023,0.000003
569,11.71465,46.07125,lamon,16,8,2023,0.000002


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/Veneto/IT_Veneto_Bins_2023.csv'

In [ ]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

In [ ]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [ ]:
##TODO Visualisation of results

,municipality,geometry,x,y,day,month,year,probability
0,chioggia,"POLYGON ((12.29589 45.33225, 12.29961 45.30769...",12.24756,45.27153,1,5,2023,0.003282
1,venezia,"POLYGON ((12.58835 45.53969, 12.58795 45.53951...",12.32478,45.43497,1,5,2023,0.002148
2,codevigo,"POLYGON ((12.12752 45.30091, 12.12959 45.30046...",12.18085,45.26512,1,5,2023,0.001883
3,rosolina,"POLYGON ((12.32883 45.14614, 12.32880 45.14586...",12.30160,45.08601,1,5,2023,0.000718
4,cavallino treporti,"POLYGON ((12.51153 45.50279, 12.51263 45.50274...",12.49633,45.47024,1,5,2023,0.000645
